# 01.2 — Behavior and composition

[Previous: 01.1](01.1-objects-and-state.ipynb) · [Class index and setup](README.md) · [Next: 01.3](01.3-schemas-and-validation.ipynb)

**Purpose:** Choose where a calculation belongs and explain how objects can work together.

**In class:** 30 minutes. **Independent work:** 15 minutes, included in the class index's 210-minute allowance. The separately assessed project adds workload.

## What you will be able to do

- Distinguish a calculation from an operation on instance state.
- Use composition and justify a method-versus-function choice.
- Recognize a small inheritance example and polymorphic use.

## Starting point

Conceptually, a Report owns a name and a list of records. This file repeats the complete implementation; no previous notebook state is needed.

This file includes all definitions and inputs it needs. Read the parts in order, but start this one with **Restart Kernel → Run All** in VS Code. Do not execute another notebook or an earlier exercise solution to prepare it. Empty answer cells are safe during Run All.

Reference solutions stay in separate cells: attempt the task, uncomment its `%load` line, and run that cell to reveal source. Inspect the result before running the populated cell separately. Keep your answer in its own cell because loading can replace the invoking cell. A loaded solution remains in the notebook even after restarting the kernel.

In [ ]:
from pathlib import Path
import sys

if not Path("solutions/exercise_01_2_e1.py").is_file():
    raise FileNotFoundError(
        "Open 01-oop-and-schemas.code-workspace in VS Code and restart the kernel. "
        "The working directory must be the class folder containing README.md and solutions/."
    )
print("Notebook kernel:", sys.executable)
print("Class files are available.")

## In class · 30–50 minutes — Behavior needs a home

A **function** can take all its inputs explicitly. A **method** can use the receiving object's state through `self`. Neither is automatically better. Ask whether the operation needs this report's state or is simply a reusable calculation over supplied records.

The function below only reads its argument. The report's `total` method supplies its own records to that function. This keeps the calculation usable even when a caller has a list but does not need a report object.

In [ ]:
def total_value(records):
    """Add values without changing the supplied records."""
    return sum(record["value"] for record in records)


class Report:
    def __init__(self, name):
        self.name = name
        self.records = []

    def add(self, record):
        self.records.append(record)

    def total(self):
        return total_value(self.records)

    def summary(self):
        return {"name": self.name, "count": len(self.records), "total": self.total()}

In [ ]:
rows = [{"label": "north", "value": 2.5}, {"label": "south", "value": 3.5}]
report = Report("daily")
for row in rows:
    report.add(row)
print("Function:", total_value(rows))
print("Method:", report.total())
print("Summary:", report.summary())
assert total_value(rows) == report.total() == 6

### Illustration — ProgrammingObject

<img src="images/object-state-and-methods.png" alt="An object is drawn as a device with attribute displays and three method controls." width="600" style="max-width: 100%; height: auto; background-color: white;">

Attributes describe current state; methods provide operations on an object. In this lesson, Report.add changes collected state while total and mean read it. The device is an analogy: a Python method need not change state, and this picture does not enforce private access.

*Credit: [ProgrammingObject](https://commons.wikimedia.org/wiki/File:ProgrammingObject.svg) by chris (Wikimedia user Chrkl), [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/). Commons PNG rendering; no content changes. [Image sources and teaching notes](IMAGE_SUGGESTIONS.md).*

### Read and change deliberately

`add` changes the report. `total_value`, `total`, and `summary` calculate results without adding records. A method can return useful data rather than immediately printing it: a terminal, test, or later web interface can decide how to display the returned dictionary.

A calculation should not silently remove records or fetch new input each time it is called. A small, explicit operation is easier to explain and check. Notice that `summary()` describes the report's current state; it does not store a second total that could become stale.

### Composition: a report has records

Our report contains a list of measurement dictionaries. This **has-a** relationship is composition: it builds behavior by holding and using other values/objects. In the next part, validated Observation objects will take the place of those dictionaries. A report is not a special kind of measurement, so making `Report` inherit from `Observation` would express the wrong relationship.

Do not build a hierarchy just because several things appear in the same dataset. Start with clear responsibilities: records describe individual observations; a report accumulates and summarizes them.

### 🧩 Exercise 01.2.E1 — A mean with a clear contract

> **In class · 10 minutes**
>
> **Goal:** Separate reusable calculation from convenient access to report state.
>
> **Start here:** The function/Report definition above. You may copy it into your answer using a new name if you want to keep the demo unchanged.
>
> **Task:** Write a mean_value(records) function and expose it through a report mean() method. Decide what an empty collection returns, using None for this exercise. Explain when a caller would use the function directly.
>
> **Check:** Measurements 2 and 4 produce 3.0. An empty report produces None without raising ZeroDivisionError.
>
> **Hint:** Handle the empty list before dividing. The method can pass self.records to the function.

In [ ]:
# Your attempt for 01.2.E1.
# Keep your code and explanation here, separate from the reference cell.

In [ ]:
# Reveal only after attempting the exercise.
# Remove the leading '# ' from the next line and run this cell.
# %load solutions/exercise_01_2_e1.py

## Guided follow-up — Independent work · 15 minutes

### Inheritance: a small recognition example

Inheritance describes a type derived from another type. A derived object can reuse behavior and replace a particular method. In this provided example, `AuditReport` inherits construction and accumulation, while its summary adds a source marker. `super().summary()` calls the base behavior first.

This example introduces vocabulary. It is not a recommendation to create a separate subclass for every report option.

In [ ]:
class AuditReport(Report):
    def summary(self):
        result = super().summary()
        result["source"] = "local"
        return result


plain = Report("plain")
audited = AuditReport("audited")
for candidate in [plain, audited]:
    print(candidate.summary())

### Polymorphism: the caller uses the same operation

The loop calls `summary()` on each object without a special branch for its concrete type. The behavior differs, while the caller uses the same operation. This is a small example of polymorphism. The extra `source` key is visible in the audited summary; both versions retain `name`, `count`, and `total`.

Composition and inheritance answer different questions. A Report **has** records; an AuditReport **is a** specialized Report in this small demonstration. Pydantic will provide another practical inheritance example: our Observation class will inherit validation behavior from its BaseModel.

### Empty data is a design decision

An empty sum naturally has total zero. An empty mean has no observations to divide, and returning zero would make it look like an observed average of zero. `None` makes absence explicit. A clear contract lets callers and checks handle this deliberately.

### Completion check and reflection

Include these questions in the 15-minute follow-up:

- Why can the total calculation be a function even though Report offers a total method?
- What state justifies keeping Report as a class?
- Identify the composition, inheritance, and polymorphic call in the examples.
- Explain why recomputing a small summary is safer here than manually maintaining another total attribute.

**Optional extension · 15 minutes, excluded from required time:** compare the subclass with a report that stores a `source` attribute. Which interface is simpler for one extra field?

## References

- [Python inheritance](https://docs.python.org/3/tutorial/classes.html#inheritance)
- [Previous: objects and state](01.1-objects-and-state.ipynb)
- [Next: schemas and validation](01.3-schemas-and-validation.ipynb)
- [Class index](README.md)